# Caracterización del conjunto preparado - MMOTU / OTU_2D

Este notebook corresponde a la caracterización del conjunto de datos obtenido después de la ejecución completa del pipeline de preparación y organización. El análisis utiliza los metadatos generados durante la ejecución para cuantificar la composición del conjunto, sus particiones, categorías y principales propiedades técnicas.

## 1. Configuración y carga de metadatos

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = (PROJECT_ROOT / "data" / "processed" / "MMOTU_OTU_2D")
METADATA_FILE = (PROCESSED_DATA_DIR / "metadata.csv")

metadata_df = pd.read_csv(METADATA_FILE)
print("Registros de metadatos:", len(metadata_df))
metadata_df.head()

Registros de metadatos: 1469


,image_id,image_name,split,original_class,binary_label,binary_class,source_image_path,source_mask_path,organized_image_path,organized_mask_path,...,image_format,mask_width,mask_height,mask_channels,mask_dtype,mask_format,mask_values,same_dimensions,status,exclusion_reason
0,658,658.JPG,train,5,0,benign,data/raw/MMOTU/OTU_2D/images/658.JPG,data/raw/MMOTU/OTU_2D/annotations/658_binary.PNG,data/processed/MMOTU_OTU_2D/train/images/658.JPG,data/processed/MMOTU_OTU_2D/train/masks/658_bi...,...,.JPG,480,559,1,uint8,.PNG,"0,1",True,included,NaN
1,384,384.JPG,train,3,0,benign,data/raw/MMOTU/OTU_2D/images/384.JPG,data/raw/MMOTU/OTU_2D/annotations/384_binary.PNG,data/processed/MMOTU_OTU_2D/train/images/384.JPG,data/processed/MMOTU_OTU_2D/train/masks/384_bi...,...,.JPG,554,589,1,uint8,.PNG,"0,1",True,included,NaN
2,367,367.JPG,train,3,0,benign,data/raw/MMOTU/OTU_2D/images/367.JPG,data/raw/MMOTU/OTU_2D/annotations/367_binary.PNG,data/processed/MMOTU_OTU_2D/train/images/367.JPG,data/processed/MMOTU_OTU_2D/train/masks/367_bi...,...,.JPG,958,534,1,uint8,.PNG,"0,1",True,included,NaN
3,730,730.JPG,train,1,0,benign,data/raw/MMOTU/OTU_2D/images/730.JPG,data/raw/MMOTU/OTU_2D/annotations/730_binary.PNG,data/processed/MMOTU_OTU_2D/train/images/730.JPG,data/processed/MMOTU_OTU_2D/train/masks/730_bi...,...,.JPG,973,547,1,uint8,.PNG,"0,1",True,included,NaN
4,1426,1426.JPG,train,7,1,malignant,data/raw/MMOTU/OTU_2D/images/1426.JPG,data/raw/MMOTU/OTU_2D/annotations/1426_binary.PNG,data/processed/MMOTU_OTU_2D/train/images/1426.JPG,data/processed/MMOTU_OTU_2D/train/masks/1426_b...,...,.JPG,958,535,1,uint8,.PNG,"0,1",True,included,NaN


## 2. Verificación de completitud de los metadatos

Se verifica que el archivo de metadatos contenga un registro por cada muestra incluida y que los campos principales necesarios para su identificación, clasificación y caracterización técnica se encuentren disponibles.

In [2]:
# Caracterización del dataset
print("Número de filas:", len(metadata_df))
print("Número de columnas:", len(metadata_df.columns))
print("\nValores faltantes por columna:")
metadata_df.isna().sum()

Número de filas: 1469
Número de columnas: 24

Valores faltantes por columna:


image_id                   0
image_name                 0
split                      0
original_class             0
binary_label               0
binary_class               0
source_image_path          0
source_mask_path           0
organized_image_path       0
organized_mask_path        0
image_width                0
image_height               0
image_channels             0
image_dtype                0
image_format               0
mask_width                 0
mask_height                0
mask_channels              0
mask_dtype                 0
mask_format                0
mask_values                0
same_dimensions            0
status                     0
exclusion_reason        1469
dtype: int64

In [3]:
required_columns = [
    "image_id",
    "image_name",
    "split",

    "original_class",
    "binary_label",
    "binary_class",

    "organized_image_path",
    "organized_mask_path",

    "image_width",
    "image_height",
    "image_channels",
    "image_dtype",
    "image_format",

    "mask_width",
    "mask_height",
    "mask_channels",
    "mask_dtype",
    "mask_format",
    "mask_values",

    "same_dimensions",
    "status",
]

missing_columns = [
    column
    for column in required_columns
    if column not in metadata_df.columns
]
print("Columnas requeridas ausentes:", missing_columns)

assert len(metadata_df) == 1469
assert len(missing_columns) == 0
assert (metadata_df["status"] == "included").all()
print("Metadatos completos para las muestras incluidas.")

Columnas requeridas ausentes: []
Metadatos completos para las muestras incluidas.


## 3. Distribución de las muestras por partición

Se cuantifica la distribución de las muestras incluidas de acuerdo con las particiones proporcionadas por el conjunto de datos original.

In [ ]:
split_distribution = (
    metadata_df["split"]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="count")
)

# Calcular el porcentaje de cada split
split_distribution["percentage"] = (split_distribution["count"] / len(metadata_df) * 100)
split_distribution

,split,count,percentage
0,train,1000,68.073519
1,validation,469,31.926481


In [5]:
# Validación de la distribución por partición
assert split_distribution["count"].sum() == 1469
assert (metadata_df["split"] == "train").sum() == 1000
assert (metadata_df["split"] == "validation").sum() == 469
print("Distribución por partición verificada correctamente.")

Distribución por partición verificada correctamente.


## 4. Distribución de las categorías originales

El conjunto OTU_2D utiliza ocho categorías originales codificadas mediante valores del 0 al 7. Por ello,se conserva y documenta la composición original del conjunto antes de analizar la agrupación binaria utilizada en la investigación.

In [6]:
# Validación de la distribución por clase original
original_class_distribution = pd.crosstab(
    metadata_df["original_class"],
    metadata_df["split"]
)
# Reindexar para asegurar que todas las clases estén presentes en el DataFrame
original_class_distribution = (
    original_class_distribution
    .reindex(
        index=range(8),
        columns=["train", "validation"],
        fill_value=0
    )
)

original_class_distribution["total"] = (original_class_distribution["train"] + original_class_distribution["validation"])
original_class_distribution

split,train,validation,total
original_class,,,
0,226,110,336
1,153,66,219
2,228,108,336
3,57,31,88
4,47,19,66
5,180,87,267
6,71,33,104
7,38,15,53


In [7]:
assert (original_class_distribution["total"].sum() == 1469)
assert (original_class_distribution["train"].sum() == 1000)
assert (original_class_distribution["validation"].sum() == 469)
print("Distribución de categorías originales verificada correctamente.")

Distribución de categorías originales verificada correctamente.


## 5. Distribución de las clases binarias

A partir de las categorías originales, se analiza la distribución correspondiente a la agrupación binaria definida para la investigación. La etiqueta binaria permite cuantificar la composición de las clases benigno y maligno en cada partición.

In [10]:
# Validación de la distribución por clase binaria
binary_distribution = pd.crosstab(
    metadata_df["binary_class"],
    metadata_df["split"]
)
# Reindexar para asegurar que todas las clases estén presentes en el DataFrame
binary_distribution = (
    binary_distribution
    .reindex(
        index=["benign", "malignant"],
        columns=["train", "validation"],
        fill_value=0
    )
)

binary_distribution["total"] = (binary_distribution["train"] + binary_distribution["validation"])
binary_distribution["percentage_total"] = (binary_distribution["total"] / len(metadata_df) * 100)
binary_distribution

split,train,validation,total,percentage_total
binary_class,,,,
benign,891,421,1312,89.312457
malignant,109,48,157,10.687543


In [11]:
# Porcentaje dentro de cada split
binary_percentage_by_split = pd.crosstab(
    metadata_df["binary_class"],
    metadata_df["split"],
    normalize="columns"
) * 100

binary_percentage_by_split = (
    binary_percentage_by_split
    .reindex(
        index=["benign", "malignant"],
        columns=["train", "validation"],
        fill_value=0
    )
)

binary_percentage_by_split

split,train,validation
binary_class,,
benign,89.1,89.765458
malignant,10.9,10.234542


In [12]:
assert binary_distribution.loc["benign", "train"] == 891
assert binary_distribution.loc["malignant", "train"] == 109
assert binary_distribution.loc["benign", "validation"] == 421
assert binary_distribution.loc["malignant", "validation"] == 48
assert binary_distribution["total"].sum() == 1469

print("Distribución binaria verificada correctamente.")

Distribución binaria verificada correctamente.


## 6. Caracterización técnica de las imágenes y máscaras

Se caracterizan las principales propiedades técnicas registradas durante la ejecución del pipeline, incluyendo dimensiones, formato, número de canales y tipo de dato de las imágenes ecográficas y sus máscaras asociadas.

In [13]:
technical_characterization = {
    "image_formats": metadata_df["image_format"].value_counts(),
    "mask_formats": metadata_df["mask_format"].value_counts(),
    "image_channels": metadata_df["image_channels"].value_counts(),
    "mask_channels": metadata_df["mask_channels"].value_counts(),
    "image_dtypes": metadata_df["image_dtype"].value_counts(),
    "mask_dtypes": metadata_df["mask_dtype"].value_counts(),
}

for name, result in technical_characterization.items():
    print(f"\n{name}:")
    print(result)


image_formats:
image_format
.JPG    1469
Name: count, dtype: int64

mask_formats:
mask_format
.PNG    1469
Name: count, dtype: int64

image_channels:
image_channels
3    1469
Name: count, dtype: int64

mask_channels:
mask_channels
1    1469
Name: count, dtype: int64

image_dtypes:
image_dtype
uint8    1469
Name: count, dtype: int64

mask_dtypes:
mask_dtype
uint8    1469
Name: count, dtype: int64


### 6.1 Variabilidad dimensional

In [ ]:
# Combinaciones distintas de dimensiones
dimension_summary = metadata_df[
    [
        "image_width",
        "image_height",
        "mask_width",
        "mask_height",
    ]
].agg([
    "min",
    "max",
    "nunique",
])

dimension_summary

,image_width,image_height,mask_width,mask_height
min,302,266,302,266
max,1135,794,1135,794
nunique,185,186,185,186


In [ ]:
# Combinaciones distintas de dimensiones
unique_image_dimensions = (metadata_df[["image_width", "image_height"]].drop_duplicates())

print("Combinaciones distintas de dimensiones:",len(unique_image_dimensions))

Combinaciones distintas de dimensiones: 859


### 6.2 Relación de aspecto

Se analiza la relación entre el ancho y el alto de las imágenes incluidas con el propósito de documentar la variabilidad geométrica del conjunto preparado.

In [16]:
# Analisis de la relación de aspecto de las imágenes
metadata_df["aspect_ratio"] = (metadata_df["image_width"] / metadata_df["image_height"])
aspect_ratio_summary = (metadata_df["aspect_ratio"].describe())

aspect_ratio_summary

count    1469.000000
mean        1.390523
std         0.430415
min         0.721386
25%         0.905838
50%         1.555735
75%         1.795886
max         2.104283
Name: aspect_ratio, dtype: float64

In [17]:
# Análisis de la orientación de las imágenes
orientation_distribution = pd.Series({
    "horizontal": (
        metadata_df["image_width"] > metadata_df["image_height"]
    ).sum(),

    "vertical": (
        metadata_df["image_width"] < metadata_df["image_height"]
    ).sum(),

    "square": (
        metadata_df["image_width"] == metadata_df["image_height"]
    ).sum(),
})

orientation_distribution

horizontal    865
vertical      603
square          1
dtype: int64

In [18]:
# Validación de la distribución por orientación
assert orientation_distribution.sum() == 1469
print("Relación de aspecto caracterizada correctamente.")

Relación de aspecto caracterizada correctamente.


### 6.3 Correspondencia dimensional entre imágenes y máscaras

In [19]:
# Validación de la correspondencia de dimensiones entre imágenes y máscaras
dimension_correspondence = (metadata_df["same_dimensions"].value_counts())
dimension_correspondence

same_dimensions
True    1469
Name: count, dtype: int64

In [ ]:
# Conteo de muestras con diferencias dimensionales
dimension_mismatch_count = (~metadata_df["same_dimensions"]).sum()
print("Muestras con diferencias dimensionales:",dimension_mismatch_count)

Muestras con diferencias dimensionales: 0


In [21]:
assert dimension_mismatch_count == 0
print("Correspondencia dimensional verificada correctamente.")

Correspondencia dimensional verificada correctamente.


### 6.4 Codificación de las máscaras de segmentación

In [22]:
mask_value_distribution = (metadata_df["mask_values"].value_counts())
mask_value_distribution

mask_values
0,1    1469
Name: count, dtype: int64

In [23]:
assert (metadata_df["mask_values"] == "0,1").all()
print("Todas las máscaras conservan la codificación binaria {0,1}.")

Todas las máscaras conservan la codificación binaria {0,1}.


## 7. Resumen de la caracterización del conjunto preparado

Se consolidan los principales resultados cuantitativos obtenidos durante la caracterización del conjunto preparado.

In [24]:
dataset_summary = pd.DataFrame([
    {
        "caracteristica": "Muestras totales",
        "resultado": len(metadata_df),
    },
    {
        "caracteristica": "Muestras de train",
        "resultado": (
            metadata_df["split"] == "train"
        ).sum(),
    },
    {
        "caracteristica": "Muestras de validation",
        "resultado": (
            metadata_df["split"] == "validation"
        ).sum(),
    },
    {
        "caracteristica": "Clases originales",
        "resultado": metadata_df[
            "original_class"
        ].nunique(),
    },
    {
        "caracteristica": "Muestras benignas",
        "resultado": (
            metadata_df["binary_class"] == "benign"
        ).sum(),
    },
    {
        "caracteristica": "Muestras malignas",
        "resultado": (
            metadata_df["binary_class"] == "malignant"
        ).sum(),
    },
    {
        "caracteristica": "Resoluciones diferentes",
        "resultado": len(
            unique_image_dimensions
        ),
    },
    {
        "caracteristica": "Imágenes JPG",
        "resultado": (
            metadata_df["image_format"] == ".JPG"
        ).sum(),
    },
    {
        "caracteristica": "Máscaras PNG",
        "resultado": (
            metadata_df["mask_format"] == ".PNG"
        ).sum(),
    },
    {
        "caracteristica": "Correspondencia imagen-máscara",
        "resultado": (
            metadata_df["same_dimensions"]
        ).sum(),
    },
])

dataset_summary

,caracteristica,resultado
0,Muestras totales,1469
1,Muestras de train,1000
2,Muestras de validation,469
3,Clases originales,8
4,Muestras benignas,1312
5,Muestras malignas,157
6,Resoluciones diferentes,859
7,Imágenes JPG,1469
8,Máscaras PNG,1469
9,Correspondencia imagen-máscara,1469
